# Abstract Shared Schema and Multi-Source Training

This notebook tests whether a more abstract shared feature space improves portability from Cell2Cell to BankChurners, and whether pooled multi-source training helps learn a common representation.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from xgboost import XGBClassifier


ROOT = Path("/home/ms990728/소정해")
RESULT_PATH = ROOT / "results" / "abstract_common_multisource_results.md"
NOTEBOOK_PATH = ROOT / "notebooks" / "comparison" / "abstract_common_multisource_experiment.ipynb"
RANDOM_STATE = 42

XGB_PARAMS = {
    "colsample_bytree": 0.8334271527847554,
    "gamma": 0.11429345433755263,
    "learning_rate": 0.06857996256539675,
    "max_depth": 3,
    "min_child_weight": 2,
    "n_estimators": 493,
    "reg_alpha": 0.2497327922401265,
    "reg_lambda": 0.9246782213565523,
    "subsample": 0.7954562418017752,
}


def load_cell2cell(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, na_values=["NA", ""], keep_default_na=True)
    df["HandsetPrice"] = pd.to_numeric(df["HandsetPrice"].replace("Unknown", pd.NA), errors="coerce")
    df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)
    return df


def load_bankchurners(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Attrition_Flag"] = df["Attrition_Flag"].map({"Existing Customer": 0, "Attrited Customer": 1}).astype(int)
    return df


def map_partner_cell(series: pd.Series) -> pd.Series:
    return series.map({"Yes": 1.0, "No": 0.0, "Unknown": np.nan})


def map_partner_bank(series: pd.Series) -> pd.Series:
    return series.map({"Married": 1.0, "Single": 0.0, "Divorced": 0.0, "Unknown": np.nan})


def map_children_cell(series: pd.Series) -> pd.Series:
    return series.map({"Yes": 1.0, "No": 0.0})


def map_children_bank(series: pd.Series) -> pd.Series:
    return np.where(series.fillna(0) > 0, 1.0, 0.0)


def build_billing2_cell(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    X = pd.DataFrame(
        {
            "monthly_billing": pd.to_numeric(df["MonthlyRevenue"], errors="coerce"),
            "total_billing": pd.to_numeric(df["TotalRecurringCharge"], errors="coerce"),
        }
    )
    y = df["Churn"].astype(int)
    return X, y


def build_billing2_bank(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    X = pd.DataFrame(
        {
            "monthly_billing": pd.to_numeric(df["Total_Trans_Amt"], errors="coerce"),
            "total_billing": pd.to_numeric(df["Credit_Limit"], errors="coerce"),
        }
    )
    y = df["Attrition_Flag"].astype(int)
    return X, y


def build_abstract_cell(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    tenure = pd.to_numeric(df["MonthsInService"], errors="coerce")
    activity = pd.to_numeric(df["MonthlyMinutes"], errors="coerce")
    monetary = pd.to_numeric(df["MonthlyRevenue"], errors="coerce")
    capacity = pd.to_numeric(df["TotalRecurringCharge"], errors="coerce")
    overage = pd.to_numeric(df["OverageMinutes"], errors="coerce")
    change_volume = pd.to_numeric(df["PercChangeMinutes"], errors="coerce").abs()
    change_amount = pd.to_numeric(df["PercChangeRevenues"], errors="coerce").abs()
    support_contacts = pd.to_numeric(df["RetentionCalls"], errors="coerce").fillna(0) + pd.to_numeric(df["CustomerCareCalls"], errors="coerce").fillna(0)
    support_flag = df["MadeCallToRetentionTeam"].map({"Yes": 1.0, "No": 0.0})
    support = support_contacts + support_flag.fillna(0)
    relationship_depth = pd.to_numeric(df["UniqueSubs"], errors="coerce").fillna(0) + pd.to_numeric(df["ActiveSubs"], errors="coerce").fillna(0)
    partner_flag = map_partner_cell(df["MaritalStatus"])
    children_flag = map_children_cell(df["ChildrenInHH"])

    X = pd.DataFrame(
        {
            "tenure": tenure,
            "age": pd.to_numeric(df["AgeHH1"], errors="coerce"),
            "partner_flag": partner_flag,
            "children_flag": children_flag,
            "relationship_depth": relationship_depth,
            "activity_volume": activity,
            "monetary_volume": monetary,
            "capacity": capacity,
            "pressure_ratio": overage / (activity + 1.0),
            "change_intensity": change_volume + change_amount,
            "support_intensity": support,
            "volume_to_capacity": monetary / (capacity + 1.0),
            "activity_per_tenure": activity / (tenure + 1.0),
            "monetary_per_tenure": monetary / (tenure + 1.0),
            "support_per_tenure": support / (tenure + 1.0),
            "relationship_per_tenure": relationship_depth / (tenure + 1.0),
            "support_to_activity": support / (activity + 1.0),
        }
    )
    y = df["Churn"].astype(int)
    return X, y


def build_abstract_bank(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    tenure = pd.to_numeric(df["Months_on_book"], errors="coerce")
    activity = pd.to_numeric(df["Total_Trans_Ct"], errors="coerce")
    monetary = pd.to_numeric(df["Total_Trans_Amt"], errors="coerce")
    capacity = pd.to_numeric(df["Credit_Limit"], errors="coerce")
    overage = pd.to_numeric(df["Avg_Utilization_Ratio"], errors="coerce")
    change_volume = (pd.to_numeric(df["Total_Ct_Chng_Q4_Q1"], errors="coerce") - 1.0).abs()
    change_amount = (pd.to_numeric(df["Total_Amt_Chng_Q4_Q1"], errors="coerce") - 1.0).abs()
    support = pd.to_numeric(df["Contacts_Count_12_mon"], errors="coerce").fillna(0) + pd.to_numeric(df["Months_Inactive_12_mon"], errors="coerce").fillna(0)
    relationship_depth = pd.to_numeric(df["Total_Relationship_Count"], errors="coerce").fillna(0)
    partner_flag = map_partner_bank(df["Marital_Status"])
    children_flag = map_children_bank(df["Dependent_count"])

    X = pd.DataFrame(
        {
            "tenure": tenure,
            "age": pd.to_numeric(df["Customer_Age"], errors="coerce"),
            "partner_flag": partner_flag,
            "children_flag": children_flag,
            "relationship_depth": relationship_depth,
            "activity_volume": activity,
            "monetary_volume": monetary,
            "capacity": capacity,
            "pressure_ratio": overage,
            "change_intensity": change_volume + change_amount,
            "support_intensity": support,
            "volume_to_capacity": monetary / (capacity + 1.0),
            "activity_per_tenure": activity / (tenure + 1.0),
            "monetary_per_tenure": monetary / (tenure + 1.0),
            "support_per_tenure": support / (tenure + 1.0),
            "relationship_per_tenure": relationship_depth / (tenure + 1.0),
            "support_to_activity": support / (activity + 1.0),
        }
    )
    y = df["Attrition_Flag"].astype(int)
    return X, y


def split_domain(X: pd.DataFrame, y: pd.Series):
    return train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE,
    )


def fit_imputer(X_train: pd.DataFrame):
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    imputer = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    return imputer, X_train_imp


def transform_imputer(imputer: SimpleImputer, X: pd.DataFrame) -> pd.DataFrame:
    X = X.replace([np.inf, -np.inf], np.nan)
    return pd.DataFrame(imputer.transform(X), columns=X.columns, index=X.index)


def fit_rank_transform(X_train: pd.DataFrame):
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    qt = QuantileTransformer(
        output_distribution="uniform",
        random_state=RANDOM_STATE,
        n_quantiles=min(len(X_train), 1000),
    )
    X_train_rank = pd.DataFrame(qt.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    return qt, X_train_rank


def transform_rank(qt: QuantileTransformer, X: pd.DataFrame) -> pd.DataFrame:
    X = X.replace([np.inf, -np.inf], np.nan)
    return pd.DataFrame(qt.transform(X), columns=X.columns, index=X.index)


def _sqrtm_psd(mat: np.ndarray, inverse: bool = False, eps: float = 1e-6) -> np.ndarray:
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.maximum(eigvals, eps)
    values = 1.0 / np.sqrt(eigvals) if inverse else np.sqrt(eigvals)
    return eigvecs @ np.diag(values) @ eigvecs.T


def fit_coral(source_train: pd.DataFrame, target_train: pd.DataFrame) -> dict:
    xs = np.asarray(source_train, dtype=float)
    xt = np.asarray(target_train, dtype=float)
    mu_s = xs.mean(axis=0, keepdims=True)
    mu_t = xt.mean(axis=0, keepdims=True)
    cs = np.cov(xs - mu_s, rowvar=False) + 1e-6 * np.eye(xs.shape[1])
    ct = np.cov(xt - mu_t, rowvar=False) + 1e-6 * np.eye(xt.shape[1])
    a = _sqrtm_psd(cs, inverse=True) @ _sqrtm_psd(ct, inverse=False)
    return {"mu_s": mu_s, "mu_t": mu_t, "a": a}


def transform_coral(X: pd.DataFrame, params: dict) -> pd.DataFrame:
    xs = np.asarray(X, dtype=float)
    aligned = (xs - params["mu_s"]) @ params["a"] + params["mu_t"]
    return pd.DataFrame(aligned, columns=X.columns, index=X.index)


def fit_xgb(X_train: pd.DataFrame, y_train: pd.Series, sample_weight=None) -> XGBClassifier:
    scale_pos_weight = int((y_train == 0).sum()) / max(int((y_train == 1).sum()), 1)
    model = XGBClassifier(
        **XGB_PARAMS,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train, y_train, sample_weight=sample_weight)
    return model


def fit_domain_classifier(X_source: pd.DataFrame, X_target: pd.DataFrame) -> dict:
    X = pd.concat([X_source, X_target], axis=0, ignore_index=True)
    y = np.array([0] * len(X_source) + [1] * len(X_target))
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    model = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        "domain_auc": roc_auc_score(y_test, prob),
        "domain_accuracy": accuracy_score(y_test, pred),
    }


def compute_metrics(y_true, prob, threshold: float = 0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


def best_f1_threshold(y_true, prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = np.array([f1_score(y_true, (prob >= t).astype(int), zero_division=0) for t in thresholds])
    idx = int(scores.argmax())
    threshold = float(thresholds[idx])
    return threshold, float(scores[idx]), compute_metrics(y_true, prob, threshold=threshold)


def md_table(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |\n"
    header += "|" + "|".join(["---"] * len(cols)) + "|\n"
    rows = []
    for _, row in df.iterrows():
        rows.append("| " + " | ".join(str(row[c]) for c in cols) + " |")
    return header + "\n".join(rows)


def balance_domain_weights(domain_ids: np.ndarray) -> np.ndarray:
    weights = np.zeros(len(domain_ids), dtype=float)
    unique = np.unique(domain_ids)
    for dom in unique:
        mask = domain_ids == dom
        weights[mask] = len(domain_ids) / max(len(unique) * max(mask.sum(), 1), 1)
    return weights


def evaluate_transfer(source_X, source_y, target_X, target_y, method: str):
    src_imp, src_train_imp = fit_imputer(source_X[0])
    tgt_imp, tgt_train_imp = fit_imputer(target_X[0])
    src_test_imp = transform_imputer(src_imp, source_X[1])
    tgt_test_imp = transform_imputer(tgt_imp, target_X[1])

    if method == "raw":
        src_train_feat = src_train_imp
        src_test_feat = src_test_imp
        tgt_train_feat = tgt_train_imp
        tgt_test_feat = tgt_test_imp
    elif method == "rank":
        src_qt, src_train_feat = fit_rank_transform(src_train_imp)
        tgt_qt, tgt_train_feat = fit_rank_transform(tgt_train_imp)
        src_test_feat = transform_rank(src_qt, src_test_imp)
        tgt_test_feat = transform_rank(tgt_qt, tgt_test_imp)
    elif method == "coral":
        src_to_tgt = fit_coral(src_train_imp, tgt_train_imp)
        tgt_to_src = fit_coral(tgt_train_imp, src_train_imp)
        src_train_feat = transform_coral(src_train_imp, src_to_tgt)
        src_test_feat = transform_coral(src_test_imp, src_to_tgt)
        tgt_train_feat = transform_coral(tgt_train_imp, tgt_to_src)
        tgt_test_feat = transform_coral(tgt_test_imp, tgt_to_src)
    else:
        raise ValueError(method)

    src_model = fit_xgb(src_train_feat, source_y[0])
    tgt_model = fit_xgb(tgt_train_feat, target_y[0])

    src_holdout_prob = src_model.predict_proba(src_test_feat)[:, 1]
    tgt_holdout_prob = tgt_model.predict_proba(tgt_test_feat)[:, 1]
    c2t_prob = src_model.predict_proba(tgt_test_imp)[:, 1]
    t2c_prob = tgt_model.predict_proba(src_test_imp)[:, 1]

    domain_metrics = fit_domain_classifier(src_train_feat, tgt_train_imp)

    return {
        "src_holdout": compute_metrics(source_y[1], src_holdout_prob),
        "tgt_holdout": compute_metrics(target_y[1], tgt_holdout_prob),
        "c2t": compute_metrics(target_y[1], c2t_prob),
        "t2c": compute_metrics(source_y[1], t2c_prob),
        "c2t_best": best_f1_threshold(target_y[1], c2t_prob),
        "t2c_best": best_f1_threshold(source_y[1], t2c_prob),
        "domain": domain_metrics,
    }


def evaluate_pooled(source_X, source_y, target_X, target_y, method: str):
    src_imp, src_train_imp = fit_imputer(source_X[0])
    tgt_imp, tgt_train_imp = fit_imputer(target_X[0])
    src_test_imp = transform_imputer(src_imp, source_X[1])
    tgt_test_imp = transform_imputer(tgt_imp, target_X[1])

    X_train = pd.concat([src_train_imp, tgt_train_imp], axis=0)
    y_train = pd.concat([source_y[0], target_y[0]], axis=0).reset_index(drop=True)
    domain_ids = np.array([0] * len(src_train_imp) + [1] * len(tgt_train_imp))
    sample_weight = balance_domain_weights(domain_ids)
    scale_pos_weight = int((y_train == 0).sum()) / max(int((y_train == 1).sum()), 1)

    if method == "raw":
        X_train_feat = X_train
        src_test_feat = src_test_imp
        tgt_test_feat = tgt_test_imp
    elif method == "rank":
        qt = QuantileTransformer(
            output_distribution="uniform",
            random_state=RANDOM_STATE,
            n_quantiles=min(len(X_train), 1000),
        )
        X_train_feat = pd.DataFrame(qt.fit_transform(X_train), columns=X_train.columns)
        src_test_feat = pd.DataFrame(qt.transform(src_test_imp), columns=src_test_imp.columns, index=src_test_imp.index)
        tgt_test_feat = pd.DataFrame(qt.transform(tgt_test_imp), columns=tgt_test_imp.columns, index=tgt_test_imp.index)
    else:
        raise ValueError(method)

    model = XGBClassifier(
        **XGB_PARAMS,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train_feat, y_train, sample_weight=sample_weight)

    src_prob = model.predict_proba(src_test_feat)[:, 1]
    tgt_prob = model.predict_proba(tgt_test_feat)[:, 1]

    return {
        "src_holdout": compute_metrics(source_y[1], src_prob),
        "tgt_holdout": compute_metrics(target_y[1], tgt_prob),
        "src_best": best_f1_threshold(source_y[1], src_prob),
        "tgt_best": best_f1_threshold(target_y[1], tgt_prob),
    }


def summarize_transfer(method: str, res: dict) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"direction": "Cell2Cell -> BankChurners", "method": method, **res["c2t"], "inverted_auc": 1 - res["c2t"]["roc_auc"]},
            {"direction": "BankChurners -> Cell2Cell", "method": method, **res["t2c"], "inverted_auc": 1 - res["t2c"]["roc_auc"]},
        ]
    )


def summarize_holdout(method: str, res: dict) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"split": "Cell2Cell holdout", "method": method, **res["src_holdout"]},
            {"split": "BankChurners holdout", "method": method, **res["tgt_holdout"]},
        ]
    )


def summarize_threshold(method: str, res: dict) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"direction": "Cell2Cell -> BankChurners", "method": method, "best_threshold": res["c2t_best"][0], "best_f1": res["c2t_best"][1], **res["c2t_best"][2]},
            {"direction": "BankChurners -> Cell2Cell", "method": method, "best_threshold": res["t2c_best"][0], "best_f1": res["t2c_best"][1], **res["t2c_best"][2]},
        ]
    )


def build_result_text(source_df, bank_df, sections):
    lines = []
    lines.append("# Abstract Shared Schema and Multi-Source Training")
    lines.append("")
    lines.append("## Setup")
    lines.append("- Source domain: Cell2Cell")
    lines.append("- Target domain: BankChurners")
    lines.append("- Reference baseline: `billing2 = monthly_billing + total_billing`")
    lines.append("- New schema: `abstract_shared`")
    lines.append("- Multi-source training: pooled Cell2Cell + BankChurners")
    lines.append("- Methods: `raw`, `rank normalization`, `CORAL` for transfer; pooled `raw`, `rank` for multi-source")
    lines.append("")
    lines.append("## Dataset Size")
    lines.append(f"- Cell2Cell rows: `{len(source_df):,}`; churn rate `{source_df['Churn'].mean():.4f}`")
    lines.append(f"- BankChurners rows: `{len(bank_df):,}`; attrition rate `{bank_df['Attrition_Flag'].mean():.4f}`")
    lines.append("")
    lines.append("## Abstract Shared Schema")
    lines.append("- tenure")
    lines.append("- age")
    lines.append("- partner_flag")
    lines.append("- children_flag")
    lines.append("- relationship_depth")
    lines.append("- activity_volume")
    lines.append("- monetary_volume")
    lines.append("- capacity")
    lines.append("- pressure_ratio")
    lines.append("- change_intensity")
    lines.append("- support_intensity")
    lines.append("- volume_to_capacity")
    lines.append("- activity_per_tenure")
    lines.append("- monetary_per_tenure")
    lines.append("- support_per_tenure")
    lines.append("- relationship_per_tenure")
    lines.append("- support_to_activity")
    lines.append("")
    lines.append("## Source-Only Transfer")
    lines.append(sections["transfer"].to_string(index=False))
    lines.append("")
    lines.append("## In-Domain Holdout")
    lines.append(sections["holdout"].to_string(index=False))
    lines.append("")
    lines.append("## Threshold Sensitivity")
    lines.append(sections["threshold"].to_string(index=False))
    lines.append("")
    lines.append("## Multi-Source Pooled Training")
    lines.append(sections["pooled"].to_string(index=False))
    lines.append("")
    lines.append("## Domain Separability")
    lines.append(sections["domain"].to_string(index=False))
    lines.append("")
    lines.append("## Interpretation")
    lines.append("- If the abstract schema were truly more universal, Cell2Cell -> BankChurners AUC should rise above the `billing2` baseline.")
    lines.append("- If pooled multi-source training improves both holdouts, then the common representation is being learned rather than merely transferred.")
    lines.append("- If domain AUC stays far from 0.5, the two domains are still easy to separate even after abstraction.")
    return "\n".join(lines)


def write_notebook(script_text: str):
    nb = {
        "cells": [
            {
                "cell_type": "markdown",
                "metadata": {},
                "source": [
                    "# Abstract Shared Schema and Multi-Source Training\n",
                    "\n",
                    "This notebook tests whether a more abstract shared feature space improves portability from Cell2Cell to BankChurners, and whether pooled multi-source training helps learn a common representation.\n",
                ],
            },
            {
                "cell_type": "code",
                "execution_count": None,
                "metadata": {},
                "outputs": [],
                "source": script_text.splitlines(keepends=True),
            },
        ],
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.11"},
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    NOTEBOOK_PATH.write_text(json.dumps(nb, ensure_ascii=False, indent=2), encoding="utf-8")


def main():
    cell_df = load_cell2cell(ROOT / "data" / "raw" / "cell2celltrain.csv")
    bank_df = load_bankchurners(ROOT / "data" / "external" / "credit_card_churn" / "BankChurners.csv")

    # Source-only transfer on the reference billing2 schema.
    cell_billing_X, cell_y = build_billing2_cell(cell_df)
    bank_billing_X, bank_y = build_billing2_bank(bank_df)
    c_X_train_b, c_X_test_b, c_y_train_b, c_y_test_b = split_domain(cell_billing_X, cell_y)
    b_X_train_b, b_X_test_b, b_y_train_b, b_y_test_b = split_domain(bank_billing_X, bank_y)

    billing_results = evaluate_transfer(
        (c_X_train_b, c_X_test_b),
        (c_y_train_b, c_y_test_b),
        (b_X_train_b, b_X_test_b),
        (b_y_train_b, b_y_test_b),
        method="coral",
    )

    # New abstract shared schema.
    cell_abs_X, cell_abs_y = build_abstract_cell(cell_df)
    bank_abs_X, bank_abs_y = build_abstract_bank(bank_df)
    c_X_train, c_X_test, c_y_train, c_y_test = split_domain(cell_abs_X, cell_abs_y)
    b_X_train, b_X_test, b_y_train, b_y_test = split_domain(bank_abs_X, bank_abs_y)

    raw_res = evaluate_transfer((c_X_train, c_X_test), (c_y_train, c_y_test), (b_X_train, b_X_test), (b_y_train, b_y_test), method="raw")
    rank_res = evaluate_transfer((c_X_train, c_X_test), (c_y_train, c_y_test), (b_X_train, b_X_test), (b_y_train, b_y_test), method="rank")
    coral_res = evaluate_transfer((c_X_train, c_X_test), (c_y_train, c_y_test), (b_X_train, b_X_test), (b_y_train, b_y_test), method="coral")

    pooled_raw = evaluate_pooled((c_X_train, c_X_test), (c_y_train, c_y_test), (b_X_train, b_X_test), (b_y_train, b_y_test), method="raw")
    pooled_rank = evaluate_pooled((c_X_train, c_X_test), (c_y_train, c_y_test), (b_X_train, b_X_test), (b_y_train, b_y_test), method="rank")

    source_only_holdout = pd.concat(
        [summarize_holdout("raw", raw_res), summarize_holdout("rank", rank_res), summarize_holdout("coral", coral_res)],
        ignore_index=True,
    )
    source_only_transfer = pd.concat(
        [summarize_transfer("raw", raw_res), summarize_transfer("rank", rank_res), summarize_transfer("coral", coral_res)],
        ignore_index=True,
    )
    source_only_threshold = pd.concat(
        [summarize_threshold("raw", raw_res), summarize_threshold("rank", rank_res), summarize_threshold("coral", coral_res)],
        ignore_index=True,
    )
    pooled_holdout = pd.concat(
        [
            summarize_holdout("pooled_raw", pooled_raw),
            summarize_holdout("pooled_rank", pooled_rank),
        ],
        ignore_index=True,
    )

    domain_df = pd.DataFrame(
        [
            {"method": "raw", **raw_res["domain"]},
            {"method": "rank", **rank_res["domain"]},
            {"method": "coral", **coral_res["domain"]},
        ]
    )

    sections = {
        "holdout": source_only_holdout,
        "transfer": source_only_transfer,
        "threshold": source_only_threshold,
        "pooled": pooled_holdout,
        "domain": domain_df,
    }
    result_text = build_result_text(cell_df, bank_df, sections)
    RESULT_PATH.write_text(result_text, encoding="utf-8")

    print("Wrote:", RESULT_PATH)
    print()
    print("=== Reference billing2 baseline (CORAL transfer) ===")
    print(md_table(pd.DataFrame(
        [
            {"direction": "Cell2Cell -> BankChurners", "roc_auc": billing_results["c2t"]["roc_auc"], "f1": billing_results["c2t"]["f1"]},
            {"direction": "BankChurners -> Cell2Cell", "roc_auc": billing_results["t2c"]["roc_auc"], "f1": billing_results["t2c"]["f1"]},
        ]
    ).round(4)))
    print()
    print("=== Abstract shared schema source-only transfer ===")
    print(md_table(source_only_transfer.round(4)))
    print()
    print("=== Abstract shared schema pooled training ===")
    print(md_table(pooled_holdout.round(4)))

    script_path = Path(__file__) if "__file__" in globals() else None
    if script_path is not None and script_path.exists():
        write_notebook(script_path.read_text(encoding="utf-8"))


if __name__ == "__main__":
    main()
